In [ ]:
# Install BERTopic and its core dependencies
!pip install bertopic sentence-transformers umap-learn hdbscan kaleido -q

import pandas as pd
import numpy as np
from bertopic import BERTopic
from sentence_transformers import SentenceTransformer
from umap import UMAP
from hdbscan import HDBSCAN
from sklearn.feature_extraction.text import CountVectorizer
import matplotlib.pyplot as plt
import seaborn as sns
from google.colab import runtime, drive

print("Libraries imported successfully!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.7/154.7 kB 12.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.0/69.0 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.3/49.3 kB 3.6 MB/s eta 0:00:00


/usr/local/lib/python3.12/dist-packages/hdbscan/robust_single_linkage_.py:175: SyntaxWarning: invalid escape sequence '\{'
  $max \{ core_k(a), core_k(b), 1/\alpha d(a,b) \}$.


Libraries imported successfully!


## Load the Phase 2 Dataset

In [ ]:
drive.mount('/content/drive')

# Load the data generated from Phase 2
# file_path = '/content/drive/MyDrive/Colab Notebooks/news_cleaned_filtered_v2.parquet'
file_path = '/content/drive/MyDrive/Colab Notebooks/df_with_entities.parquet'
print(f"Loading data from {file_path}...")
df = pd.read_parquet(file_path)

print(f"Dataset loaded! Shape: {df.shape}")

# Ensure we have our cleaned text available
docs = df['cleaned_text_v2'].astype(str).tolist()
print(f"Prepared {len(docs):,} documents for topic modeling.")

Mounted at /content/drive
Loading data from /content/drive/MyDrive/Colab Notebooks/news_cleaned_filtered_v2.parquet...
Dataset loaded! Shape: (165086, 13)
Prepared 165,086 documents for topic modeling.


## COnfigure and train BERT Topic

In [ ]:
print("Step 1/5: Loading Embedding Model...")
# all-MiniLM-L6-v2 is the industry standard for fast, high-quality English embeddings
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

print("Step 2/5: Configuring UMAP (Dimensionality Reduction)...")
# Reduces vector size so clustering works efficiently.
umap_model = UMAP(n_neighbors=15, n_components=5, min_dist=0.0, metric='cosine', random_state=42)

print("Step 3/5: Configuring HDBSCAN (Clustering)...")
# min_cluster_size=100 ensures we don't get hyper-specific topics with only 5 articles.
hdbscan_model = HDBSCAN(min_cluster_size=100, metric='euclidean', cluster_selection_method='eom', prediction_data=True)

print("Step 4/5: Configuring Vectorizer (Stop Word Removal)...")
# Removes common English words so topics are defined by actual AI/Business terms
# ngram_range=(1, 2) allows for two-word phrases like "machine learning" or "data privacy"
vectorizer_model = CountVectorizer(stop_words="english", ngram_range=(1, 2))

print("Step 5/5: Initializing and Training BERTopic...")
topic_model = BERTopic(
    embedding_model=embedding_model,
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    vectorizer_model=vectorizer_model,
    language="english",
    verbose=True,
    calculate_probabilities=False 
)

# Fit the model 
topics, probs = topic_model.fit_transform(docs)

# Add the topic ID back to  main dataframe
df['Topic_ID'] = topics

print("Topic Modeling Complete!")

Step 1/5: Loading Embedding Model...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

2026-03-09 20:03:18,046 - BERTopic - Embedding - Transforming documents to embeddings.


Step 2/5: Configuring UMAP (Dimensionality Reduction)...
Step 3/5: Configuring HDBSCAN (Clustering)...
Step 4/5: Configuring Vectorizer (Stop Word Removal)...
Step 5/5: Initializing and Training BERTopic...


Batches:   0%|          | 0/5159 [00:00<?, ?it/s]

2026-03-09 21:36:53,009 - BERTopic - Embedding - Completed ✓
2026-03-09 21:36:53,011 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-03-09 21:42:23,038 - BERTopic - Dimensionality - Completed ✓
2026-03-09 21:42:23,043 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-03-09 21:42:51,979 - BERTopic - Cluster - Completed ✓
2026-03-09 21:42:52,029 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-03-09 21:51:19,201 - BERTopic - Representation - Completed ✓


✅ Topic Modeling Complete!


## Review and Interpret Topics

In [ ]:
# Extract the topic information
topic_info = topic_model.get_topic_info()

# Note: Topic -1 is the "Outlier" topic. These are articles that didn't fit into any tight cluster.
# It is normal for Topic -1 to be the largest group in HDBSCAN clustering.
print("Top 15 Discovered Topics:")
display(topic_info.head(16))

# Let's map the auto-generated names to our dataframe for easier reading later
topic_mapping = topic_info.set_index('Topic')['Name'].to_dict()
df['Topic_Name'] = df['Topic_ID'].map(topic_mapping)

Top 15 Discovered Topics:


,Topic,Count,Name,Representation,Representative_Docs
0,-1,68872,-1_ai_news_new_said,"[ai, news, new, said, 2023, technology, data, ...",[Europe agreed on world-leading AI rules. How ...
1,0,4702,0_healthcare_health_patient_medical,"[healthcare, health, patient, medical, clinica...",[Cloud-Based AI Endoscopy System Assists Gastr...
2,1,3429,1_data_ai_generative ai_generative,"[data, ai, generative ai, generative, cloud, e...","[Oracle unveils new suite for analytics, AI da..."
3,2,3409,2_schedule_npr_radio_donate,"[schedule, npr, radio, donate, programs, arts,...",[Research finds how AI will impact demographic...
4,3,2393,3_chatgpt_gpt_openai_users,"[chatgpt, gpt, openai, users, model, chatbot, ...",[Capabilities of OpenAI's GPT-4 Language Model...
5,4,2160,4_india_indian_ai_delhi,"[india, indian, ai, delhi, news, global, minis...","[Delhi launches AI innovation drive for 500,00..."
6,5,1939,5_india_2025_vs_viral,"[india, 2025, vs, viral, check, watch video, w...",[Business News | Protectt.ai Launches New Vers...
7,6,1723,6_crypto_blockchain_bitcoin_presale,"[crypto, blockchain, bitcoin, presale, ozak, d...",[Exclusive interview with Virtuals Protocol co...
8,7,1603,7_china_deepseek_chinese_baidu,"[china, deepseek, chinese, baidu, alibaba, ai,...","[What is DeepSeek, the Chinese AI company upen..."
9,8,1531,8_menafn_research_currencies_europe arab,"[menafn, research, currencies, europe arab, ma...",[Level Up With AI Stocks - (Nasdaq: ALAB) (NAS...


## Visualize

In [11]:
%pip install -U kaleido

In [ ]:
# Generate a polished bar chart of the top 8 topics (excluding the -1 outlier group)
fig = topic_model.visualize_barchart(top_n_topics=8, n_words=5, width=300, height=300)

# Display in notebook
fig.show()

# fig.write_image("/content/drive/MyDrive/Colab Notebooks/top_ai_topics_barchart.png", scale=2)
fig.write_html("/content/drive/MyDrive/Colab Notebooks/top_ai_topics_barchart.html")
print("Saved visualization to top_ai_topics_barchart.png")

Saved visualization to top_ai_topics_barchart.png


In [16]:
fig_topic = topic_model.visualize_topics()

# Display in notebook
fig_topic.show()
fig_topic.write_html("/content/drive/MyDrive/Colab Notebooks/top_ai_topics_distance_map.html")

In [ ]:
print("Reducing overlapping topics...")
# Force the model to merge similar clusters until only 30 distinct topics remain.
topic_model.reduce_topics(docs, nr_topics=30)

# Check the new, consolidated topics
new_topic_info = topic_model.get_topic_info()
display(new_topic_info.head(10))

# Update  dataframe with the new consolidated topic IDs
df['Topic_ID'] = topic_model.topics_

Reducing overlapping topics...


2026-03-09 22:18:39,079 - BERTopic - Topic reduction - Reducing number of topics
2026-03-09 22:18:39,238 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-03-09 22:28:28,680 - BERTopic - Representation - Completed ✓
2026-03-09 22:28:31,049 - BERTopic - Topic reduction - Reduced number of topics from 245 to 30


,Topic,Count,Name,Representation,Representative_Docs
0,-1,68872,-1_ai_news_new_data,"[ai, news, new, data, technology, said, 2023, ...",[Top 12 Machine Learning Use Cases and Busines...
1,0,38094,0_ai_new_news_data,"[ai, new, news, data, best, technology, 2023, ...",[Definition of AI prompt | PCMag Skip to Main ...
2,1,15646,1_nasdaq_ai_market_stocks,"[nasdaq, ai, market, stocks, data, news, stock...",[1 Artificial Intelligence (AI) Stock Down 44%...
3,2,12582,2_news_ai_said_public,"[news, ai, said, public, schedule, new, radio,...","[PayPal, Venmo users to gain early access to P..."
4,3,5223,3_health_healthcare_ai_medical,"[health, healthcare, ai, medical, clinical, pa...","[On Tuesday, Microsoft Corp. announced advance..."
5,4,4803,4_price_market_rate_ai,"[price, market, rate, ai, crypto, trading, mex...",[AIW to PKR: Convert Stability World AI to Pak...
6,5,4736,5_india_ai_news_2025,"[india, ai, news, 2025, world, vs, delhi, indi...",[Global AI Race: India Emerges As World’s 3rd ...
7,6,4516,6_rawpixel_px 300_jpeg_px,"[rawpixel, px 300, jpeg, px, 300, image rawpix...",[Sky backgrounds outdoors nature. AI | Premium...
8,7,1587,7_energy_ai_data_power,"[energy, ai, data, power, new, climate, learni...",[Artificial Intelligence Will Be Critical For ...
9,8,1158,8_automotive_vehicle_vehicles_ai,"[automotive, vehicle, vehicles, ai, car, engin...",[OLEDWorks Announces Launch of New Automotive ...


In [19]:
fig_topic = topic_model.visualize_topics()
fig_topic.show()

fig_topic.write_html("/content/drive/MyDrive/Colab Notebooks/top_ai_topics_reduced_distance_map.html")

### Save for next steps

In [ ]:
output_path = '/content/drive/MyDrive/Colab Notebooks/df_with_topics_and_entities.parquet'
df.to_parquet(output_path, index=False)

#Save the actual BERTopic model 
topic_model.save("/content/drive/MyDrive/Colab Notebooks/bertopic_ai_model", serialization="safetensors", save_ctfidf=True)

print(f"Phase 3 Complete. Data saved to {output_path}")
print("Ready for Phase 4: Sentiment Analysis!")

Phase 3 Complete. Data saved to /content/drive/MyDrive/Colab Notebooks/df_with_topics_and_entities.parquet
Ready for Phase 4: Sentiment Analysis!


In [ ]:
print("All tasks complete and saved to Google Drive.")
print("Shutting down the Colab runtime to save compute credits...")

# This command forcefully unassigns the GPU and terminates the session
runtime.unassign()

All tasks complete and saved to Google Drive.
Shutting down the Colab runtime to save compute credits...
